In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

import ripser
import persim

In [2]:
########## read the data ##########

processes = [f"run_{i:02d}_decayed_1" for i in range(1,12)]
kappa_3 = [0.8, 0.84, 0.88, 0.92, 0.96, 1, 1.04, 1.08, 1.12, 1.16, 1.2]
# Xsection_NoDecay = [3.7154566e-01, 3.617386e-01, 3.524371e-01, 3.4375437e-01, 3.3552887e-01, 3.273126e-01, 3.197475e-01, 3.126318e-01, 3.058909e-01, 2.993453e-01, 2.935034e-01]     ### hhvv unit: fb
Xsection_NoDecay = [5.5479319e-2, 5.433959e-2, 5.3013911e-2, 5.1688287e-2, 5.06150778e-2, 4.9453757e-2, 4.833858e-2, 4.740145e-2, 4.6342168e-2, 4.543942e-2, 4.4229705e-2]     ### hhmumu unit: fb
BR_hbb = 0.5824
Xsection = []
for i in range(11):
    Xsection.append(Xsection_NoDecay[i]*BR_hbb*BR_hbb)

Luminosity = 10000   ### unit: fb^-1
simulation_num = 100000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Eta", "VLCjetR10N2.Phi", "VLCjetR10N2.Mass", "MissingET.MET", "VLCjetR10N2.BTag", "EFlow.Eta", "EFlow.Phi", "EFlow.ET"]

##### set data path #####

event_path = []
for type in processes:
    event_path.append("/data/mucollider/two_boosted/10TeV/scan_kappa3_hhmumu_small/Events/" + type + "/delphes_events.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print(process, "Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

run_01_decayed_1 Time: 3.7947 (s)
run_02_decayed_1 Time: 6.3127 (s)
run_03_decayed_1 Time: 8.8750 (s)
run_04_decayed_1 Time:11.4718 (s)
run_05_decayed_1 Time:13.8628 (s)
run_06_decayed_1 Time:16.5023 (s)
run_07_decayed_1 Time:19.1979 (s)
run_08_decayed_1 Time:21.6073 (s)
run_09_decayed_1 Time:24.4163 (s)
run_10_decayed_1 Time:26.8864 (s)
run_11_decayed_1 Time:29.3300 (s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### define X_HH (defined in 2207.09602v2) #####

def calculate_X_HH(event):
    jet_mass1 = event[features.index("VLCjetR10N2.Mass")][0]
    jet_mass2 = event[features.index("VLCjetR10N2.Mass")][1]
    diff1 = np.abs(jet_mass1 - 124)
    diff2 = np.abs(jet_mass2 - 124)
    if diff1<diff2:
        m1 = jet_mass1
        m2 = jet_mass2
    else:
        m1 = jet_mass2
        m2 = jet_mass1
    return np.sqrt(((m1-124)/(0.1*m1+0.00001))**2 + ((m2-115)/(0.1*m2+0.00001))**2)

##### calculate M_JJ #####

def calculate_M_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    M_JJ = np.sqrt(p[0]**2 - p[1]**2 - p[2]**2 - p[3]**2)
    return M_JJ

##### calculate pT_JJ #####

def calculate_pT_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    pT_JJ = np.sqrt(p[1]**2 + p[2]**2)
    return pT_JJ

##### calculate \Delta_\Eta #####

def calculate_dEta_JJ(event):
    return np.abs(event[features.index("VLCjetR10N2.Eta")][0] - event[features.index("VLCjetR10N2.Eta")][1])

##### calculate TDA #####

def calculate_TDA(event):
#     neutrino_ID = [12, 14, 16, -12, -14, -16]
#     where_jet_particle = np.where(event[features.index("Particle.Status")]==1)[0]
#     where_neutrino = np.where(np.isin(event[features.index("Particle.PID")], neutrino_ID))[0]
#     where_jet_particle = np.delete(where_jet_particle, np.isin(where_jet_particle, where_neutrino))

    particle_ET = event[features.index("EFlow.ET")]
    particle_ETcut = 0
    where_ET_larger = np.where(particle_ET>particle_ETcut)
    particle_ET = event[features.index("EFlow.ET")][where_ET_larger]
    
    center_eta = (event[features.index("VLCjetR10N2.Eta")][0]+event[features.index("VLCjetR10N2.Eta")][1])/2    ### centerize the event
    center_phi = (event[features.index("VLCjetR10N2.Phi")][0]+event[features.index("VLCjetR10N2.Phi")][1])/2
    particle_eta = event[features.index("EFlow.Eta")][where_ET_larger]-center_eta
    particle_phi = event[features.index("EFlow.Phi")][where_ET_larger]-center_phi
    
    where_larger_pi = np.where(particle_phi>np.pi)
    particle_phi[where_larger_pi] = -(2*np.pi - particle_phi[where_larger_pi])    ### check if jet particle split
    where_smaller_mpi = np.where(particle_phi<-np.pi)
    particle_phi[where_smaller_mpi] = 2*np.pi + particle_phi[where_smaller_mpi]    ### check if jet particle split

    P = np.array([particle_eta, particle_phi]).T   ### calculate TDA
    diagrams = ripser.ripser(P)['dgms']
    
    return diagrams

##### calculate TDA sum of lifetime #####

def TDA_sum(diagrams):
    sum_l0 = np.sum(diagrams[0][:-1,1] - diagrams[0][:-1,0])
    sum_l1 = np.sum(diagrams[1][:,1] - diagrams[1][:,0])
    return sum_l0, sum_l1

##### calculate TDA mean of lifetime #####

def TDA_mean(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    if np.sum(l1)==0:
        l1 = np.concatenate([l1,[0]])
    return np.mean(l0), np.mean(l1)

##### calculate TDA entropy #####

def TDA_entropy(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    L0 = np.sum(l0)
    S0 = (l0/L0)*np.log2((l0/L0))
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    L1 = np.sum(l1)
    S1 = (l1/L1)*np.log2((l1/L1))
    return -np.sum(S0), -np.sum(S1)

##### calculate TDA Lifetime*Birthtime #####

def TDA_LB(diagrams):
    return np.sum(diagrams[1][:,0]*(diagrams[1][:,1]-diagrams[1][:,0]))

In [4]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 800   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 100000 run_01_decayed_1 events. Corresponding cross section: 0.018818016896573444 (fb)
There are 100000 run_02_decayed_1 events. Corresponding cross section: 0.0184314324905984 (fb)
There are 100000 run_03_decayed_1 events. Corresponding cross section: 0.01798177574875136 (fb)
There are 100000 run_04_decayed_1 events. Corresponding cross section: 0.017532137662341123 (fb)
There are 100000 run_05_decayed_1 events. Corresponding cross section: 0.01716811609136333 (fb)
There are 100000 run_06_decayed_1 events. Corresponding cross section: 0.01677420796792832 (fb)
There are 100000 run_07_decayed_1 events. Corresponding cross section: 0.016395951348940802 (fb)
There are 100000 run_08_decayed_1 events. Corresponding cross section: 0.016078086449152002 (fb)
There are 100000 run_09_decayed_1 events. Corresponding cross section: 0.015718788841799683 (fb)
There are 100000 run_10_decayed_1 events. Corresponding cross section: 0.015412585964339203 (fb)
There are 100

In [5]:
##### calculate the high level features #####

num_of_events = []

m_J_leading = []
m_J_subleading = []
pt_J_leading = []
pt_J_subleading = []
missET = []
X_HH = []
M_JJ = []
pT_JJ = []
dEta_JJ = []
BTag_leading = []
BTag_subleading = []

for process in processes:
    num_of_events.append(len(events[processes.index(process)]))
    for i in range(len(events[processes.index(process)])):
        m_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][0])
        m_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][1])
        pt_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][0])
        pt_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][1])
        missET.append(events[processes.index(process)][i][features.index("MissingET.MET")][0])
        X_HH.append(calculate_X_HH(events[processes.index(process)][i]))
        M_JJ.append(calculate_M_JJ(events[processes.index(process)][i]))
        pT_JJ.append(calculate_pT_JJ(events[processes.index(process)][i]))
        dEta_JJ.append(calculate_dEta_JJ(events[processes.index(process)][i]))
        BTag_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][0])
        BTag_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][1])
    print("Time:{:^8.4f}(s)".format(time.time()-t1))
    
print("Number of events:", num_of_events)

Time:48.3007 (s)
Time:51.0605 (s)
Time:53.8089 (s)
Time:56.5549 (s)
Time:59.2835 (s)
Time:62.0217 (s)
Time:64.7396 (s)
Time:67.4131 (s)
Time:70.0350 (s)
Time:72.6544 (s)
Time:75.2595 (s)
Number of events: [25443, 24960, 24841, 24786, 24597, 24593, 24216, 24052, 23548, 23350, 23284]


In [6]:
##### calculate TDA high level features #####

H_0 = []    ### sum of 0'th order lifetime 
H_1 = []    ### sum of 1'th order lifetime 
S_0 = []    ### sum of 0'th order entropy
S_1 = []    ### sum of 1'th order entropy
LB_1 = []   ### sum of 1'th order Lifetime*Birthtime

for process in processes:
    for i in range(len(events[processes.index(process)])):
        if (i%20000)==19999:
            print("check point")
        TDA_tmp = calculate_TDA(events[processes.index(process)][i])
        H_0.append(TDA_sum(TDA_tmp)[0])        
        H_1.append(TDA_sum(TDA_tmp)[1])
        S_0.append(TDA_entropy(TDA_tmp)[0])   
        S_1.append(TDA_entropy(TDA_tmp)[1])
        LB_1.append(TDA_LB(TDA_tmp))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

check point
Time:151.8663(s)
check point
Time:226.8345(s)
check point
Time:301.3620(s)
check point
Time:376.0704(s)
check point
Time:450.1138(s)
check point
Time:524.0724(s)
check point
Time:596.9583(s)
check point
Time:669.3920(s)
check point
Time:740.1085(s)
check point
Time:810.1307(s)
check point
Time:880.2925(s)


In [7]:
##### make target #####

target_kappa_3 = np.repeat(kappa_3, num_of_events)

In [8]:
##### output test data #####

d = {'m_J_leading':m_J_leading, 'm_J_subleading':m_J_subleading, 'pt_J_leading':pt_J_leading, 'pt_J_subleading':pt_J_subleading, 'missET':missET, 'X_HH':X_HH, 'M_JJ':M_JJ, 'pT_JJ':pT_JJ, 'dEta_JJ':dEta_JJ, 'BTag_leading':BTag_leading, 'BTag_subleading':BTag_subleading, 'H_0':H_0, 'H_1':H_1, 'S_0':S_0, 'S_1':S_1, 'LB_1':LB_1, 'target_kappa_3':target_kappa_3}
df = pd.DataFrame(data=d)
df

,m_J_leading,m_J_subleading,pt_J_leading,pt_J_subleading,missET,X_HH,M_JJ,pT_JJ,dEta_JJ,BTag_leading,BTag_subleading,H_0,H_1,S_0,S_1,LB_1,target_kappa_3
0,117.202827,86.584534,524.466248,312.933929,211.478653,3.332663,850.314741,219.884766,0.412544,6,7,7.201423,0.068768,4.772464,2.800203,0.009423,0.8
1,118.362480,124.982468,338.800629,240.516968,160.326508,0.294758,936.840072,168.528504,1.970478,6,4,10.127811,0.163924,5.465800,2.437843,0.027680,0.8
2,124.184555,99.229088,539.527832,367.421295,354.412872,1.589412,870.183056,363.984426,0.280936,6,6,6.451117,0.144522,5.194679,1.376368,0.035103,0.8
3,127.720886,95.367157,478.522888,332.712128,171.572052,2.079168,1481.219024,145.876583,2.429255,7,7,12.163547,0.092799,4.279997,2.950491,0.011445,0.8
4,118.175781,119.232780,492.215393,487.226837,95.538452,0.481744,1296.149167,94.579347,1.515165,6,7,10.349452,0.140855,5.379471,2.736150,0.024243,0.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267665,108.174744,123.479706,677.148010,467.260101,179.239182,0.632352,2429.821273,217.593297,2.797870,6,7,13.525717,0.181320,5.127177,2.229856,0.028513,1.2
267666,104.449020,120.603607,269.847931,212.680801,128.224274,1.048676,786.507183,127.072069,2.060191,6,1,11.471275,0.117953,5.526107,2.727403,0.028233,1.2
267667,115.957916,125.614433,488.727509,333.572540,495.871918,0.152782,940.586717,496.210891,1.427320,7,5,11.863505,0.084342,5.252497,2.608977,0.014828,1.2
267668,102.746910,75.218391,553.656067,353.851257,210.710999,5.678919,3101.265793,201.297232,3.848213,7,6,16.467336,0.345584,5.042838,3.250138,0.432628,1.2


In [9]:
h5f = h5py.File('/data/mucollider/two_boosted/10TeV/scan_kappa3_hhmumu_small_HL.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")

已寫入 Dataset: m_J_leading
已寫入 Dataset: m_J_subleading
已寫入 Dataset: pt_J_leading
已寫入 Dataset: pt_J_subleading
已寫入 Dataset: missET
已寫入 Dataset: X_HH
已寫入 Dataset: M_JJ
已寫入 Dataset: pT_JJ
已寫入 Dataset: dEta_JJ
已寫入 Dataset: BTag_leading
已寫入 Dataset: BTag_subleading
已寫入 Dataset: H_0
已寫入 Dataset: H_1
已寫入 Dataset: S_0
已寫入 Dataset: S_1
已寫入 Dataset: LB_1
已寫入 Dataset: target_kappa_3


In [10]:
print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:882.0416(s)


In [11]:
h5f.close()